In [ ]:
Spark starten

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("BDLC Mini Test") \
    .getOrCreate()

spark

26/05/15 10:55:56 WARN Utils: Your hostname, bdlc-021 resolves to a loopback address: 127.0.1.1; using 10.176.129.84 instead (on interface ens3)
26/05/15 10:55:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 10:55:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/15 10:55:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [ ]:
Test Dataset erstellen

In [2]:
data = [
    ("2024-01-01 08:15:00", "2024-01-01 08:35:00", 3.2, 18.50, 2.00),
    ("2024-01-01 17:10:00", "2024-01-01 17:45:00", 5.1, 27.00, 3.50),
    ("2024-01-02 11:00:00", "2024-01-02 11:12:00", 2.0, 12.00, 1.00),
    ("2024-01-02 23:30:00", "2024-01-02 23:50:00", 7.5, 31.00, 4.00),
]

columns = [
    "pickup_datetime",
    "dropoff_datetime",
    "trip_distance_km",
    "total_amount",
    "tip_amount"
]

df = spark.createDataFrame(data, columns)

df.show()
df.printSchema()

26/05/15 10:56:07 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/05/15 10:56:19 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/15 10:56:34 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/15 10:56:49 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
                                                                                

+-------------------+-------------------+----------------+------------+----------+
|    pickup_datetime|   dropoff_datetime|trip_distance_km|total_amount|tip_amount|
+-------------------+-------------------+----------------+------------+----------+
|2024-01-01 08:15:00|2024-01-01 08:35:00|             3.2|        18.5|       2.0|
|2024-01-01 17:10:00|2024-01-01 17:45:00|             5.1|        27.0|       3.5|
|2024-01-02 11:00:00|2024-01-02 11:12:00|             2.0|        12.0|       1.0|
|2024-01-02 23:30:00|2024-01-02 23:50:00|             7.5|        31.0|       4.0|
+-------------------+-------------------+----------------+------------+----------+

root
 |-- pickup_datetime: string (nullable = true)
 |-- dropoff_datetime: string (nullable = true)
 |-- trip_distance_km: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)



In [ ]:
Testing, Datentypen und neue Spalte

In [ ]:
df_clean = df \
    .withColumn("pickup_datetime", F.to_timestamp("pickup_datetime")) \
    .withColumn("dropoff_datetime", F.to_timestamp("dropoff_datetime")) \
    .withColumn("hour", F.hour("pickup_datetime")) \
    .withColumn("weekday", F.dayofweek("pickup_datetime")) \
    .withColumn(
        "trip_duration_minutes",
        (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60
    ) \
    .withColumn(
        "avg_speed_kmh",
        F.col("trip_distance_km") / (F.col("trip_duration_minutes") / 60)
    ) \
    .withColumn(
        "tip_percentage",
        F.col("tip_amount") / F.col("total_amount") * 100
    )

df_clean.show()
df_clean.printSchema()

In [ ]:
Analyse

In [ ]:
hourly_analysis = df_clean.groupBy("hour").agg(
    F.count("*").alias("number_of_trips"),
    F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_min"),
    F.round(F.avg("avg_speed_kmh"), 2).alias("avg_speed_kmh"),
    F.round(F.avg("tip_percentage"), 2).alias("avg_tip_percentage")
).orderBy("hour")

hourly_analysis.show()

In [ ]:
output_path = "../02_results/mini_taxi_test.parquet"

df_clean.write.mode("overwrite").parquet(output_path)

In [ ]:
Einlesen

In [ ]:
df_loaded = spark.read.parquet(output_path)

df_loaded.show()
df_loaded.printSchema()

In [ ]:
Gespeicherte Datei suchen

In [ ]:
import os

os.getcwd()

In [ ]:
os.listdir("../02_results")

In [ ]:
print(output_path)

In [ ]:
Spark Stoppen

In [3]:
spark.stop()

In [ ]:
spark.sparkContext._jsc.sc().isStopped()